In [25]:
# Hurling Match Data - Cleaning Template

# Cleans raw CSV files of tagged hurling matches and produces processed files ready for analysis.
# Handles missing values intentionally rather than dropping rows or letting NaNs affect results.

# Edit MATCH CONFIG for each new match.
# Processed files are save to data/processed/.

In [26]:
import pandas as pd
import numpy as np

In [27]:
# MATCH CONFIG
# Edit this block for each match.

MATCH_LABEL = '2025_AIHF_CORK_TIPP'
TEAM_A = 'Cork'
TEAM_B = 'Tipperary'

# Raw file paths - update if necessary
PO_RAW_PATH = f'https://raw.githubusercontent.com/tdraths/hurling-match-data/refs/heads/main/data/raw/puckouts/puckouts_{MATCH_LABEL}.csv'
FR_RAW_PATH = f'https://raw.githubusercontent.com/tdraths/hurling-match-data/refs/heads/main/data/raw/frees/frees_{MATCH_LABEL}.csv'

# Processed output paths
PO_OUT_PATH = f'https://raw.githubusercontent.com/tdraths/hurling-match-data/refs/heads/main/data/processed/puckouts/puckouts_{MATCH_LABEL}.csv'
FR_OUT_PATH = f'https://raw.githubusercontent.com/tdraths/hurling-match-data/refs/heads/main/data/processed/frees/frees_{MATCH_LABEL}.csv'

# Goalkeeper dictionary — add an entry for every match/team combination
# Format: (match_label, team_name) -> goalkeeper_name
GOALKEEPER_MAP = {
    ('2025_AIF_COR_TIP',  'Cork'):      'Patrick Collins',
    ('2025_AIF_COR_TIP',  'Tipperary'): 'Rhys Shelly',
    ('2025_AISF_KIK_TIP', 'Kilkenny'):  'Eoin Murphy',
    ('2025_AISF_KIK_TIP', 'Tipperary'): 'Rhys Shelly',
}

# Free-taker dictionary — add designated takers
# Format: (match_label, team_name) -> player_name
# Leave out teams where frees are shared or unknown
FREE_TAKER_MAP = {
    ('2025_AISF_KIK_TIP', 'Kilkenny'): 'TJ Reid',
}

# END CONFIG

In [28]:
# LOOKUP TABLES

# Target Zones
# DL/DC/DR = short (defensive zones)
# ML/MC/MR = medium (midfield zones)
# AL/AC/AR = long (attacking zones)
ZONE_TO_TYPE = {
    'DL': 'short',  'DC': 'short',  'DR': 'short',
    'ML': 'medium', 'MC': 'medium', 'MR': 'medium',
    'AL': 'long',   'AC': 'long',   'AR': 'long',
}

# Possession Outcomes
# Used as the target variable for the Expected Points model
# Positive = pucking team scored, Negative = opposition scored
POSS_OUTCOME_VALUE = {
    'point':     1,
    'goal':      3,
    'no-score':  0,
    'opp-point': -1,
    'opp-goal':  -3,
}

# Getting Game State
def get_game_state(diff):
    """
    Converts a numberic score differential into a readable game state.

    diff is from the pucking team's perspective:
    positive    = pucking team winning
    negative    = pucking team losing
    zero        = level

    Using 4+ as a threshold because in hurling a 4-point
    lead is generally considered comfortable.
    """
    if diff <= -4:      return 'Losing 4+'
    elif diff <= -1:    return 'Losing 1-3'
    elif diff == 0:     return 'Level'
    elif diff <= 3:     return 'Winning 1-3'
    else:               return 'Winning 4+'

In [29]:
# PUCKOUT CLEANING FUNCTION

def clean_puckouts(path, match_label, team_a, team_b):
    """
    Loads and cleans a raw puckout CSV export.

    Steps:
      1. Load and standardise team names.
      2. Fix data types (minute, half)
      3. Standardise missing values
      4. Derive new columns (type, goalkeeper, game_state)
      5. Handle touch chain data to distinguish between a chain that ends versus no recorded chain
      6. Sort and re-sequence IDs
      7. Return cleaned dataframe

    Parameters
    path        : str - path to the raw CSV
    match_label : str - match identifier
    team_a      : str - Team A name
    team_b      : str - Team B name

    Returns pandas.DataFrame - cleaned puckout data
    """

    print(f"\n{'='*55}")
    print(f"  Cleaning puckouts: {match_label}")
    print(f"{'='*55}")

    # STEP 1: Load
    po = pd.read_csv(path, low_memory=False)
    print(f"  Loaded {len(po)} rows from {path}")

    # STEP 2:
    for col in ['pucking_team', 'team_a', 'team_b']:
        po[col] = po[col].str.strip().str.title()

    # STEP 3: Fix data types
    po['minute'] = pd.to_numeric(po['minute'], errors='coerce')
    po['half']   = pd.to_numeric(po['half'], errors='coerce')

    bad_minutes = po[po['minute'].isna() | (po['minute'] > 100)]
    if len(bad_minutes) > 0:
        print(f"\n  ⚠ {len(bad_minutes)} rows with suspect minute values:")
        print(bad_minutes[['id','minute','pucking_team','target_zone']].to_string())
        print("  → Fix these manually before proceeding.\n")
    else:
        print("  ✓ All minute values look valid")

    # STEP 4: Standardise missing values
    categorical_cols = ['type', 'delivery', 'target_zone', 'retained', 'next_action']

    for col in categorical_cols:
        if col in po.columns:
            po[col] = po[col].fillna('unknown')
            po[col] = po[col].replace('', 'unknown')

    po['notes'] = po['notes'].fillna('')

    if 'poss_outcome' in po.columns:
        po['poss_outcome'] = po['poss_outcome'].fillna('')
    else:
        po['poss_outcome'] = ''

    # STEP 5: Handle touch chain
    chain_act_cols = ['t1_action', 't2_action', 't3_action']
    chain_team_cols = ['t1_team', 't2_team', 't3_team']
    chain_all_cols = chain_act_cols + chain_team_cols + ['shot_outcome']

    for col in chain_all_cols:
        if col not in po.columns:
            po[col] = ''

    po[chain_all_cols] = po[chain_all_cols].fillna('')

    po['has_chain'] = (po['t1_action'] != '') & (po['t1_action'] != 'unknown')

    def fill_post_shot(row):
        """
        If a shot occurred at touch N, marks touches N+1 onwards as 'n/a' to distinguish structrual absence from missing data.

        Works through touches 1, 2, 3 in order. As soon as it finds a shot, it fills everything after it and stops.
        """

        if not row['has_chain']:
            return row

        shot_at = None
        for n in [1, 2, 3]:
            if row[f't{n}_action'] == 'shot':
                shot_at = n
                break

        if shot_at is not None:
            for n in range(shot_at + 1, 4):
                row[f't{n}_action'] = 'n/a'
                row[f't{n}_team'] = 'n/a'

        return row

    po = po.apply(fill_post_shot, axis=1)

    def get_chain_length(row):
        if not row['has_chain']:
            return 0
        count = 0
        for n in [1, 2, 3]:
            val = row[f't{n}_action']
            if val != '' and val != 'n/a':
                count = n
        return count

    po['chain_length'] = po.apply(get_chain_length, axis=1)

    print(f"  ✓ Touch chain: {po['has_chain'].sum()} of {len(po)} rows have chain data")
    chain_dist = po[po['has_chain']]['chain_length'].value_counts().sort_index()
    for length, count in chain_dist.items():
        print(f"      chain_length={length}: {count} rows")

    # STEP 6: Derive new columns
    po['type'] = po['target_zone'].map(ZONE_TO_TYPE).fillna('unknown')

    if 'type' in po.columns:
        mismatches = po[
            (po['target_zone'] != 'unknown') &
            (po['type'] != po['target_zone'].map(ZONE_TO_TYPE).fillna('unknown'))
        ]
        if len(mismatches) > 0:
            print(f"\n  ⚠ {len(mismatches)} type/zone mismatches found (type will be overridden):")
            print(mismatches[['id','minute','target_zone','type']].to_string())

    po['match_label'] = match_label
    po['opposition'] = np.where(
        po['pucking_team'] == team_a, team_b, team_a)
    po['goalkeeper'] = po.apply(
        lambda row: GOALKEEPER_MAP.get(
            (match_label, row['pucking_team']), 'unknown'
        ), axis=1)
    po['retained_bool'] = po['retained'].map({'yes': True, 'no': False})
    po['game_state'] = po['score_diff'].apply(get_game_state)
    po['net_score_value'] = po['poss_outcome'].map(POSS_OUTCOME_VALUE)
    po['outcome_logged'] = po['poss_outcome'] != ''

    print(f"  ✓ Derived columns added")
    print(f"  ✓ Possession outcome logged: "
          f"{po['outcome_logged'].sum()} of {len(po)} rows")#

    # STEP 7: Sort and re-sequence IDs
    po = po.sort_values(['half', 'minute']).reset_index(drop=True)
    po['id'] = range(1, len(po) + 1)

    # STEP 8: Validation Report
    print(f"\n CLEANED SUMMARY")
    print(f" {'-'*45}")
    print(f"  Rows:               {len(po)}")
    print(f"  Teams:              {po['pucking_team'].unique()}")
    print(f"  Minute range:       {po['minute'].min():.0f} – {po['minute'].max():.0f}")
    print(f"  Unknown zones:      {(po['target_zone']=='unknown').sum()}")
    print(f"  Unknown delivery:   {(po['delivery']=='unknown').sum()}")
    print(f"  Unknown retained:   {(po['retained']=='unknown').sum()}")
    print(f"  Retention rate:     {po['retained_bool'].mean()*100:.1f}%")
    print(f"  {'─'*45}")

    return po

In [30]:
def clean_frees(path, match_label, team_a, team_b):
    """
    Loads and cleans a raw frees/65s CSV.
    Steps:
      1. Standardise team names and data types
      2. Handle missing values
      3. Assign designated free-takers from dictionary
      4. Add match context
      5. Derive scored and possession flags
      6. Sort and re-sequence

    Parameters
    path        : str  — path to the raw CSV
    match_label : str  — match identifier
    team_a      : str  — Team A name
    team_b      : str  — Team B name

    Returns pandas.DataFrame - cleaned frees data
    """

    print(f"\n{'='*55}")
    print(f"  Cleaning frees: {match_label}")
    print(f"{'='*55}")

    fr = pd.read_csv(path, low_memory=False)
    print(f"  Loaded {len(fr)} rows from {path}")

    # STEP 1: Standardise team names
    for col in ['shooting_team', 'team_a', 'team_b']:
        fr[col] = fr[col].str.strip().str.title()

    fr['minute'] = pd.to_numeric(fr['minute'], errors='coerce')
    fr['half']   = pd.to_numeric(fr['half'],   errors='coerce')

    bad_minutes = fr[fr['minute'].isna() | (fr['minute'] > 100)]
    if len(bad_minutes) > 0:
        print(f"\n  ⚠ {len(bad_minutes)} rows with suspect minute values:")
        print(bad_minutes[['id','minute','shooting_team','outcome']].to_string())
    else:
        print("  ✓ All minute values look valid")

    # STEP 2: Handle missing values
    categorical_cols = ['set_piece_type', 'position_zone', 'attempt_type', 'striking_side', 'under_pressure', 'outcome']
    for col in categorical_cols:
        if col in fr.columns:
            fr[col] = fr[col].fillna('unknown').replace('', 'unknown')

    fr['shooter'] = fr['shooter'].fillna('').astype(str)
    fr['notes'] = fr['notes'].fillna('')

    # STEP 3: Assign free-takers
    for (ml, team), player in FREE_TAKER_MAP.items():
        if ml == match_label:
            mask = fr['shooting_team'] == team
            fr.loc[mask, 'shooter'] = player
            print(f"  ✓ Assigned {player} to {mask.sum()} {team} frees")

    # STEP 4: Add match context
    fr['match_label'] = match_label
    fr['opposition'] = np.where(
        fr['shooting_team'] == team_a, team_b, team_a
    )
    fr['game_state'] = fr['score_diff'].apply(get_game_state)

    # STEP 5: Derive shooting and possession flags
    fr['scored'] = fr['outcome'].isin(['point', 'goal'])
    fr['possession'] = fr['outcome'].isin(['point', 'goal', 'retained'])
    fr['score_value'] = fr['outcome'].map(
        {'point': 1, 'goal': 3}
    ).fillna(0)

    # STEP 6: Sort and re-sequence
    fr = fr.sort_values(['half', 'minute']).reset_index(drop=True)
    fr['id'] = range(1, len(fr) + 1)

    # Validation Report
    print(f"\n  CLEANED SUMMARY")
    print(f"  {'─'*45}")
    print(f"  Rows:               {len(fr)}")
    print(f"  Teams:              {fr['shooting_team'].unique()}")
    print(f"  Types:              {fr['set_piece_type'].value_counts().to_dict()}")
    print(f"  Overall conversion: {fr['scored'].mean()*100:.1f}%")
    for team in fr['shooting_team'].unique():
        sub = fr[fr['shooting_team']==team]
        print(f"  {team}: {sub['scored'].mean()*100:.1f}% "
              f"({sub['scored'].sum()}/{len(sub)})")
    print(f"  {'─'*45}")

    return fr

In [31]:
# RUN CLEANING

if __name__ == '__main__':

    from google.colab import files

    po_clean = clean_puckouts(PO_RAW_PATH, MATCH_LABEL, TEAM_A, TEAM_B)
    fr_clean = clean_frees(FR_RAW_PATH, MATCH_LABEL, TEAM_A, TEAM_B)

    po_clean.to_csv(f'processed_puckouts_{MATCH_LABEL}.csv', index=False)
    fr_clean.to_csv(f'processed_frees_{MATCH_LABEL}.csv', index=False)

    files.download(f'processed_puckouts_{MATCH_LABEL}.csv')
    files.download(f'processed_frees_{MATCH_LABEL}.csv')


  Cleaning puckouts: 2025_AIHF_CORK_TIPP
  Loaded 73 rows from https://raw.githubusercontent.com/tdraths/hurling-match-data/refs/heads/main/data/raw/puckouts/puckouts_2025_AIHF_CORK_TIPP.csv

  ⚠ 2 rows with suspect minute values:
    id  minute pucking_team target_zone
22  24     NaN    Tipperary          AL
62  65     NaN         Cork          AL
  → Fix these manually before proceeding.

  ✓ Touch chain: 66 of 73 rows have chain data
      chain_length=1: 7 rows
      chain_length=2: 7 rows
      chain_length=3: 52 rows
  ✓ Derived columns added
  ✓ Possession outcome logged: 73 of 73 rows

 CLEANED SUMMARY
 ---------------------------------------------
  Rows:               73
  Teams:              ['Tipperary' 'Cork']
  Minute range:       1 – 75
  Unknown zones:      0
  Unknown delivery:   1
  Unknown retained:   0
  Retention rate:     68.5%
  ─────────────────────────────────────────────

  Cleaning frees: 2025_AIHF_CORK_TIPP
  Loaded 32 rows from https://raw.githubuserconten

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>